In [ ]:
# 1. yolov10 se detect traffic light
# 2. biggest area of traffic light choose kiya..closest 

# 3. small cnn red green yellow detect ke liye train kiya

#aur ha gemini ne code likha mostly

In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model


In [ ]:
# Load both models
detector = YOLO('yolov10n.pt')
classifier = load_model('traffic_light_cnn_vertical.h5')
class_names = ['Green', 'Red', 'Yellow']

I0000 00:00:1772443298.705501  716914 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9543 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3080 Ti, pci bus id: 0000:01:00.0, compute capability: 8.6


In [ ]:
img_name ='5'
img_path = 'images_to_check/'+img_name+'.png'
img = cv2.imread(img_path)

img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
results = detector(img)[0]
img_h, img_w, _ = img.shape


0: 384x640 2 persons, 4 cars, 1 truck, 3 traffic lights, 34.5ms
Speed: 18.7ms preprocess, 34.5ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640)


In [ ]:
lights = [b for b in results.boxes if int(b.cls) == 9]


In [ ]:
if lights:
    best_box = max(lights, key=lambda b: (b.xyxy[0][2]-b.xyxy[0][0]) * (b.xyxy[0][3]-b.xyxy[0][1]))
    x1, y1, x2, y2 = map(int, best_box.xyxy[0])
    
    # 3. Crop and Preprocess for CNN
    crop = results.orig_img[y1:y2, x1:x2]
    crop = cv2.resize(crop, (64, 128))   #64,128 for vertical
    crop = crop / 255.0  # Normalize
    crop = np.expand_dims(crop, axis=0) # Add batch dimension
    
    # 4. Classify Color
    prediction = classifier.predict(crop)
    color = class_names[np.argmax(prediction)]
    
    print(f"Nearest Light Color: {color}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 326ms/step
Nearest Light Color: Red


I0000 00:00:1772443334.587190  717044 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


In [ ]:
color_map = {
    'Green': (0, 255, 0),    # Pure Green
    'Red': ( 0, 0,255 ),      # Pure Red
    'Yellow': ( 0,255, 255)  # Mixture of Red + Green
}

In [ ]:
if lights :
    conf = float(best_box.conf[0])
    box_bgr = color_map.get(color, (255, 255, 255)) # Default to white
    # 4. Draw the bounding box manually
    # (image, start_point, end_point, color_bgr, thickness)
    cv2.rectangle(results.orig_img ,(x1, y1), (x2, y2), box_bgr, 3)
    
    # Add a label
    label = f"Confidence level: {conf:.2f}"
    # plt.text(x1, y1-20, label, color='white', weight='bold', 
    #              bbox=dict(facecolor=np.array(box_bgr)/255, alpha=0.8))
    # plt.figure(figsize=(12, 8))
    # plt.imshow(img_rgb)
    # plt.axis('off')
    # # plt.title(f"Targeting: {color if lights else 'No Light Found'} ,{label}")
    # plt.show()
    # Show the result
    cv2.imshow("Selected Traffic Light  " + color + ' '+ label, results.orig_img)
    cv2.waitKey(0)
    cv2.destroyAllWindows



QFontDatabase: Cannot find font directory /home/rutwik/miniconda3/envs/tf/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/rutwik/miniconda3/envs/tf/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/rutwik/miniconda3/envs/tf/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/rutwik/miniconda3/envs/tf/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font dire